In [2]:
import sys
print(sys.executable)

C:\Users\munta\anaconda3\envs\risk-copilot\python.exe


In [4]:
import os
import duckdb
import pandas as pd
import yfinance as yf
from datetime import datetime
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [5]:
import duckdb
import os

# Create data folders
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

# Connect to DuckDB warehouse
con = duckdb.connect('../data/financial_warehouse.duckdb')
print("DuckDB warehouse created successfully!")

DuckDB warehouse created successfully!


In [6]:
# Define our universe of companies
tickers = [
    'AAPL',  # Apple
    'MSFT',  # Microsoft
    'JPM',   # JPMorgan Chase
    'BAC',   # Bank of America
    'GS',    # Goldman Sachs
    'AMZN',  # Amazon
    'TSLA',  # Tesla
    'XOM',   # ExxonMobil
    'JNJ',   # Johnson & Johnson
    'WMT'    # Walmart
]

# Download 5 years of daily price data
print("Downloading market data...")
market_data = yf.download(tickers, start='2019-01-01', end='2024-12-31', group_by='ticker')
print(f"Downloaded data shape: {market_data.shape}")
print("Done!")

[*********************100%***********************]  10 of 10 completed

Downloaded data shape: (1509, 50)
Done!


In [7]:
# Flatten and clean the data
all_data = []

for ticker in tickers:
    df = market_data[ticker].copy()
    df['ticker'] = ticker
    df['date'] = df.index
    df = df.reset_index(drop=True)
    df.columns = [c.lower().replace(' ', '_') for c in df.columns]
    all_data.append(df)

combined = pd.concat(all_data, ignore_index=True)
combined = combined.dropna()

# Save to DuckDB
con.execute("DROP TABLE IF EXISTS market_prices")
con.execute("""
    CREATE TABLE market_prices AS 
    SELECT * FROM combined
""")

# Verify
result = con.execute("SELECT ticker, COUNT(*) as rows FROM market_prices GROUP BY ticker").df()
print(result)

  ticker  rows
0    XOM  1509
1   AAPL  1509
2    JPM  1509
3   TSLA  1509
4     GS  1509
5    JNJ  1509
6    WMT  1509
7   AMZN  1509
8   MSFT  1509
9    BAC  1509


In [8]:
# Compute rolling metrics using SQL inside DuckDB
con.execute("""
    CREATE OR REPLACE TABLE risk_features AS
    SELECT
        ticker,
        date,
        close,
        volume,
        -- Daily return
        (close - LAG(close) OVER (PARTITION BY ticker ORDER BY date)) 
            / LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS daily_return,
        -- 30-day rolling volatility
        STDDEV(close) OVER (
            PARTITION BY ticker 
            ORDER BY date 
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS volatility_30d,
        -- 7-day moving average
        AVG(close) OVER (
            PARTITION BY ticker 
            ORDER BY date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS ma_7d,
        -- 30-day moving average
        AVG(close) OVER (
            PARTITION BY ticker 
            ORDER BY date 
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ) AS ma_30d
    FROM market_prices
    ORDER BY ticker, date
""")

# Preview
df_features = con.execute("SELECT * FROM risk_features LIMIT 10").df()
print(df_features)

  ticker       date      close     volume  daily_return  volatility_30d  \
0   AAPL 2019-01-02  37.503719  148158800           NaN             NaN   
1   AAPL 2019-01-03  33.768070  365248800     -0.099607        2.641503   
2   AAPL 2019-01-04  35.209610  234428400      0.042689        1.883970   
3   AAPL 2019-01-07  35.131245  219111200     -0.002226        1.548899   
4   AAPL 2019-01-08  35.800949  164101200      0.019063        1.353131   
5   AAPL 2019-01-09  36.408928  180396400      0.016982        1.267970   
6   AAPL 2019-01-10  36.525295  143122800      0.003196        1.205194   
7   AAPL 2019-01-11  36.166679  108092800     -0.009818        1.124840   
8   AAPL 2019-01-14  35.622841  129756800     -0.015037        1.054125   
9   AAPL 2019-01-15  36.351921  114843600      0.020467        1.009431   

       ma_7d     ma_30d  
0  37.503719  37.503719  
1  35.635895  35.635895  
2  35.493800  35.493800  
3  35.403161  35.403161  
4  35.482719  35.482719  
5  35.637087  35.6

In [11]:
# Check all tables in warehouse
tables = con.execute("SHOW TABLES").df()
print("Tables in warehouse:")
print(tables)

# Check row counts
for table in tables['name']:
    count = con.execute(f"SELECT COUNT(*) as rows FROM {table}").df()
    print(f"{table}: {count['rows'][0]} rows")

con.close()
print("\nWarehouse saved to data/financial_warehouse.duckdb")

Tables in warehouse:
            name
0  market_prices
1  risk_features
market_prices: 15090 rows
risk_features: 15090 rows

Warehouse saved to data/financial_warehouse.duckdb
